# Chapter 14: Design YouTube
*From "System Design Interview"*

## TL;DR

Design a video streaming platform like YouTube. The system has two core flows: **video uploading** (ingest, transcode, store, distribute to CDN) and **video streaming** (serve from CDN edge servers via adaptive bitrate protocols). The transcoding pipeline uses a **DAG model** for parallel, flexible video processing. Key optimizations target **speed** (chunked parallel uploads, message queues), **safety** (pre-signed URLs, DRM), and **cost** (long-tail CDN strategy).

## Requirements

### Functional
- Upload videos (max 1 GB)
- Stream/watch videos with adaptive quality
- Support mobile apps, web browsers, smart TVs

### Non-Functional
- Fast upload speed
- Smooth streaming with quality switching
- Low infrastructure cost
- High availability, scalability, reliability
- 5M DAU

## Back-of-the-Envelope Estimation

In [ ]:
# YouTube-like system estimation
dau = 5_000_000
videos_watched_per_day = 5
upload_ratio = 0.10           # 10% of users upload 1 video/day
avg_video_size_mb = 300

# Storage
daily_storage_tb = dau * upload_ratio * avg_video_size_mb / 1e6
print(f"Daily upload storage: {daily_storage_tb:.0f} TB")

# CDN cost (assuming $0.02/GB, 100% US traffic)
cost_per_gb = 0.02
daily_streaming_gb = dau * videos_watched_per_day * (avg_video_size_mb / 1000)
daily_cdn_cost = daily_streaming_gb * cost_per_gb
print(f"Daily CDN cost:      ${daily_cdn_cost:,.0f}")
print(f"Monthly CDN cost:    ${daily_cdn_cost * 30:,.0f}")

# Storage growth per year
yearly_storage_pb = daily_storage_tb * 365 / 1000
print(f"\nYearly storage:      {yearly_storage_pb:.1f} PB")

## High-Level Design

```mermaid
graph TB
    subgraph Clients
        M[Mobile]
        W[Web]
        TV[Smart TV]
    end

    subgraph Upload Flow
        LB[Load Balancer]
        API[API Servers]
        OS[Original Storage Blob]
        TS[Transcoding Servers]
        TrS[Transcoded Storage]
        CDN2[CDN]
        CQ[Completion Queue]
        CH[Completion Handler]
        MDB[Metadata DB + Cache]
    end

    M & W & TV --> LB
    LB --> API
    API --> OS
    OS --> TS
    TS --> TrS
    TrS --> CDN2
    TS --> CQ
    CQ --> CH
    CH --> MDB

    subgraph Stream Flow
        CDN[CDN Edge Servers]
    end

    M & W & TV -.-> CDN
```

| Component | Purpose |
|-----------|---------|
| **Original Storage** | Blob storage for raw uploaded videos |
| **Transcoding Servers** | Convert to multiple formats/resolutions |
| **CDN** | Edge-cached video delivery |
| **Completion Queue** | Message queue for async transcoding events |
| **Metadata DB/Cache** | Video metadata, user data |

## Deep Dive: Video Transcoding Architecture

### Why Transcode?
- Raw video is huge (hundreds of GB per hour at 60fps HD)
- Different devices need different formats/codecs
- Adaptive bitrate requires multiple resolution versions
- Container formats (.mp4, .avi) + Codecs (H.264, VP9, HEVC)

### DAG Processing Model

```mermaid
graph LR
    V[Original Video] --> S{Split}
    S --> VID[Video Stream]
    S --> AUD[Audio Stream]
    S --> META[Metadata]

    VID --> INS[Inspection]
    VID --> ENC[Video Encoding 720p 1080p 4K]
    VID --> THUMB[Thumbnail]
    VID --> WM[Watermark]

    AUD --> AENC[Audio Encoding]

    ENC --> OUT[Encoded Output]
    AENC --> OUT
    THUMB --> OUT
    META --> OUT
```

Tasks within a stage run in **parallel**; stages execute **sequentially**.

## Deep Dive: Transcoding Components

| Component | Responsibility |
|-----------|---------------|
| **Preprocessor** | Split video into GOPs, generate DAG, cache segments |
| **DAG Scheduler** | Split DAG into task stages, enqueue to resource manager |
| **Resource Manager** | Task queue + worker queue + running queue + scheduler |
| **Task Workers** | Execute encoding, thumbnailing, watermarking, etc. |
| **Temporary Storage** | Cache metadata (memory) and video/audio (blob) during processing |

## Deep Dive: System Optimizations

### Speed Optimizations
| Optimization | How |
|---|---|
| **Parallel upload** | Client splits video into GOP-aligned chunks for resumable upload |
| **Geo upload centers** | Multiple upload centers (CDN) close to users globally |
| **Message queues** | Decouple pipeline stages for parallel execution |

### Safety Optimizations
| Optimization | How |
|---|---|
| **Pre-signed URLs** | Client fetches a time-limited upload URL from API; only authorized uploads |
| **DRM** | Apple FairPlay, Google Widevine, Microsoft PlayReady |
| **AES encryption** | Encrypt video, decrypt on authorized playback |
| **Visual watermarking** | Overlay identifying info on video |

### Cost Optimizations
| Optimization | How |
|---|---|
| **Long-tail CDN** | Only popular videos on CDN; others from origin servers |
| **On-demand encoding** | Short/unpopular videos encoded only when requested |
| **Regional CDN** | Do not distribute regionally popular content globally |
| **ISP partnerships** | Build your own CDN nodes at ISP edge (Netflix Open Connect) |

## Trade-offs

| Decision | Pro | Con |
|----------|-----|-----|
| DAG model for transcoding | Flexible, parallel pipelines | Complex orchestration |
| Pre-signed upload URLs | Secure, no proxy bottleneck | Extra round-trip for URL |
| GOP-aligned chunked upload | Resumable, parallelizable | Client-side complexity |
| CDN for all videos | Lowest latency | Very expensive at scale |
| CDN for popular only (long-tail) | Cost-efficient | Higher latency for unpopular |
| Message queues between stages | Loose coupling, parallelism | Added queue infrastructure |

## Key Takeaways

1. **Two distinct flows**: upload (write-heavy, async) and streaming (read-heavy, CDN)
2. **DAG model** gives flexibility to define custom video processing pipelines with parallelism
3. **CDN cost dominates** -- optimize with long-tail strategy and ISP partnerships
4. **Pre-signed URLs** are the standard pattern for secure direct-to-storage uploads
5. **Message queues** between pipeline stages enable loose coupling and independent scaling

## See Also
- [[video-uploading-flow]] | [[video-transcoding]] | [[video-streaming]]
- [[dag-model]] | [[video-system-optimizations]]